# Loading libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import linkage, dendrogram

# Loading dataframe

In [ ]:
df = pd.read_csv('')
df 

In [ ]:
df.info()

# K-Means

In [ ]:
drop_cols = ['']
kmeans_df = df.drop(columns = drop_cols)

k_means_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('cluster', KMeans(n_clusters = 4, random_state = 30))
])

k_means_pipeline.fit(kmeans_df)

In [ ]:
interia = []
for k in range(2, 16):
    k_means_pipeline_temp = Pipeline([
    ('scaler', StandardScaler()),
    ('cluster', KMeans(n_clusters = k, n_init = 'auto', random_state = 30))
    ])
    k_means_pipeline_temp.fit(kmeans_df)
    interia.append(k_means_pipeline_temp.named_steps['cluster'].interia_)

In [ ]:
inertia_series = pd.Series(interia, index = range(2, 16))

plt.figure(figsize = (12, 8))
sns.lineplot(data = inertia_series,
             markers = 'o')
plt.title('Number of K clusters VS Interia')
plt.xlabel('K Clusters')
plt.ylabel('Interia')

In [ ]:
cluster_labels = pd.Series(k_means_pipeline.named_steps['cluster'].labels, name = 'Cluster')

kmeans_df = pd.concat([kmeans_df, cluster_labels], axis = 1)

fig = plt.figure(figsize = (12, 8))
ax = Axes3D(fig)
fig.add_axes(ax)

sc = ax.scatter(kmeans_df., kmeans_df., kmeans_df.,
                c = kmeans_df.Cluster,
                cmap = 'Set2')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_zlabel('')

plt.legend(*sc.legend_elements(), title = 'clusters', bbox_to_anchor = (1.05, 1))

In [ ]:
cluster_centers = pd.DataFrame(k_means_pipeline.named_steps['cluster'].cluster_centers_, columns = kmeans_df.columns)

cluster_centers

In [ ]:
plt.figure(figsize = (12, 8))
sns.heatmap(cluster_centers,
            cmap = 'RdBu',
            annot = True,
            fmt = '.2f')
plt.title('Centroids')

# Hierarchical

In [ ]:
hier_df = df.drop(columns = drop_cols)

hierarchical_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

hierarchical_pipeline.fit_transform(hier_df)

In [ ]:
linkage_matrix = linkage(hier_df, method = 'ward')
dendro = dendrogram(linkage_matrix)

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Data points')
plt.ylabel('Euclidean Distance')

# DBSCAN

In [ ]:
dbscan_df = df.drop(columns = drop_cols)

dbscan_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('cluster', DBSCAN(eps = .5, min_samples = 5))
])

dbscan_pipeline.fit(dbscan_df)

In [ ]:
silhouette_scores = []
eps_range = np.arange(0.3, 2.0, 0.1)
min_samples_range = [3, 5, 10, 15]

for eps in eps_range:
    for min_samples in min_samples_range:
        dbscan_pipeline_temp = Pipeline([
            ('scaler', StandardScaler()),
            ('cluster', DBSCAN(eps=eps, min_samples=min_samples))
        ])
        
        dbscan_pipeline_temp.fit(dbscan_df)
        labels = dbscan_pipeline_temp.named_steps['cluster'].labels_
        
        score = silhouette_score(dbscan_df, labels)
        silhouette_scores.append({'eps': eps, 'min_samples': min_samples, 'silhouette': score})

silhouette_df = pd.DataFrame(silhouette_scores)
print(f"Best silhouette score: {silhouette_df['silhouette'].max():.4f}")
print(silhouette_df.loc[silhouette_df['silhouette'].idxmax()])

In [ ]:
plt.figure(figsize=(14, 8))
for min_samp in silhouette_df['min_samples'].unique():
    data = silhouette_df[silhouette_df['min_samples'] == min_samp].sort_values('eps')
    sns.lineplot(data=data, x='eps', y='silhouette', marker='o', label=f'min_samples={min_samp}')

plt.title('DBSCAN Silhouette Score Optimization')
plt.xlabel('eps')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
cluster_labels = pd.Series(dbscan_pipeline.named_steps['cluster'].labels_, name = 'Cluster')

dbscan_df = pd.concat([dbscan_df, cluster_labels], axis = 1)

fig = plt.figure(figsize = (12, 8))
ax = Axes3D(fig)
fig.add_axes(ax)

sc = ax.scatter(dbscan_df., dbscan_df., dbscan_df.,
                c = dbscan_df.Cluster,
                cmap = 'Set2')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_zlabel('')

plt.legend(*sc.legend_elements(), title = 'clusters', bbox_to_anchor = (1.05, 1))